# Test run: cyclic peptide binder vs PDL1 (no AlphaFold 3)

This notebook mirrors the **PDL1** example from the project README (programmed death-ligand 1 binder design). The README’s primary worked example is a protein–protein binder against the extracellular domain sequence shown in the Boltz section.

## AlphaFold 3 and commercial use

Google DeepMind’s **AlphaFold 3** terms restrict certain uses (including many commercial applications). This notebook **does not** invoke AF3: we omit `--use_alphafold3_validation` and `--use_msa_for_af3`. Design and in-pipeline scoring still use **Boltz** (diffusion + structure prediction) and the rest of the Protein Hunter stack as implemented in `boltz_ph/`.

After design, this notebook can run **Chai-1 cross-validation** on high-ipTM YAMLs (`--downstream_validation chai`), i.e. re-predict complexes with Chai (not Boltz) while keeping the same **PyRosetta** post-processing as AF3 validation. Set `DOWNSTREAM_VALIDATION = "none"` in the command cell to skip CV for a faster smoke test. Chai weights should be available (see README / `./setup.sh`).

## Prerequisites

- Run cells with the Jupyter **kernel working directory** inside this repo (or any subfolder); `find_repo_root` walks parents to locate `boltz_ph/design.py`.
- The subprocess uses **`cwd=REPO_ROOT`** (repo root), not `boltz_ph/`. The pipeline calls LigandMPNN with `./LigandMPNN/run.py`, which is relative to the process cwd—matching the README pattern `python boltz_ph/design.py` from the repo root.
- GPU recommended; override with env var `PROTEIN_HUNTER_GPU` or edit `GPU_ID` in the code cell.
- Conda env / `./setup.sh` completed per README.

## 1. Paths and target sequence

Canonical **PDL1** target sequence from the README Boltz examples (single-chain protein target; design chain is **A**, target **B** in pipeline conventions).

In [1]:
import os
import subprocess
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Walk upward from start until boltz_ph/design.py exists."""
    for p in [start, *start.parents]:
        if (p / "boltz_ph" / "design.py").is_file():
            return p
    raise FileNotFoundError(
        "Could not find boltz_ph/design.py. Open the notebook from Protein-Hunter "
        "or set REPO_ROOT manually to the repo root."
    )


# Jupyter: kernel cwd is usually the folder containing the notebook
REPO_ROOT = find_repo_root(Path.cwd())

DESIGN_SCRIPT = REPO_ROOT / "boltz_ph" / "design.py"

# README Boltz example: human PD-L1 extracellular region (one-letter code)
PDL1_TARGET_SEQ = (
    "AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKVQHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVNAPYAAALE"
)

GPU_ID = int(os.environ.get("PROTEIN_HUNTER_GPU", "0"))

print("REPO_ROOT:", REPO_ROOT)
print("design.py:", DESIGN_SCRIPT)
print("PDL1 length:", len(PDL1_TARGET_SEQ))

REPO_ROOT: /home/kyle/Protein-Hunter
design.py: /home/kyle/Protein-Hunter/boltz_ph/design.py
PDL1 length: 122


## 2. Build a minimal **cyclic peptide** design command (Boltz)

This follows the README’s **cyclic peptide binder** recipe: short length window, `--cyclic`, high `iptm` threshold. Values are reduced for a quick **smoke test** (`num_designs`, `num_cycles`); increase them for real campaigns.

**Intentionally omitted** (AF3 / AF3-input-related):

- `--use_alphafold3_validation`
- `--use_msa_for_af3`

**Cross-validation (Boltz designs):** the next cell sets `DOWNSTREAM_VALIDATION` to `"chai"` (Chai re-folds high-ipTM YAMLs) or `"none"`. On a **Chai** design notebook you would use `--downstream_validation boltz` instead; that flag is not used here because this path runs `boltz_ph/design.py`.

In [2]:
RUN_NAME = "notebook_PDL1_cyclic_peptide_no_af3_smoke"
OUT_DIR = REPO_ROOT / "outputs" / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Cross-validate Boltz-designed hits with Chai-1 (true out-of-stack CV). Use "none" to skip.
DOWNSTREAM_VALIDATION = "chai"  # "none" | "chai"

cmd = [
    "python",
    str(DESIGN_SCRIPT),
    "--num_designs", "1",
    "--num_cycles", "3",
    "--protein_seqs", PDL1_TARGET_SEQ,
    "--msa_mode", "mmseqs",
    "--gpu_id", str(GPU_ID),
    "--name", RUN_NAME,
    "--min_protein_length", "10",
    "--max_protein_length", "20",
    "--high_iptm_threshold", "0.8",
    "--percent_X", "100",
    "--save_dir", str(OUT_DIR),
    "--cyclic",
    "--plot",
]
if DOWNSTREAM_VALIDATION and DOWNSTREAM_VALIDATION != "none":
    cmd += ["--downstream_validation", DOWNSTREAM_VALIDATION]

print("Working directory for subprocess (repo root):", REPO_ROOT)
print("DOWNSTREAM_VALIDATION:", DOWNSTREAM_VALIDATION)
print("Command:\n", " \\\n".join(cmd))

Working directory for subprocess (repo root): /home/kyle/Protein-Hunter
DOWNSTREAM_VALIDATION: chai
Command:
 python \
/home/kyle/Protein-Hunter/boltz_ph/design.py \
--num_designs \
1 \
--num_cycles \
3 \
--protein_seqs \
AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKVQHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVNAPYAAALE \
--msa_mode \
mmseqs \
--gpu_id \
0 \
--name \
notebook_PDL1_cyclic_peptide_no_af3_smoke \
--min_protein_length \
10 \
--max_protein_length \
20 \
--high_iptm_threshold \
0.8 \
--percent_X \
100 \
--save_dir \
/home/kyle/Protein-Hunter/outputs/notebook_PDL1_cyclic_peptide_no_af3_smoke \
--cyclic \
--plot \
--downstream_validation \
chai


## 3. Run design

Run with **`cwd=REPO_ROOT`**. Python still puts `boltz_ph/` on `sys.path` as the script directory for `design.py`, while `./LigandMPNN/run.py` resolves correctly under the repo root.

In [3]:
proc = subprocess.run(
    cmd,
    cwd=str(REPO_ROOT),
    env={**os.environ},
    check=False,
)
print("Exit code:", proc.returncode)
if proc.returncode != 0:
    raise RuntimeError("design.py failed; scroll up for stderr/stdout from the subprocess.")

Design Configuration:
gpu_id                        : 0
grad_enabled                  : False
name                          : notebook_PDL1_cyclic_peptide_no_af3_smoke
mode                          : binder
num_designs                   : 1
num_cycles                    : 3
cyclic                        : True
min_protein_length            : 10
max_protein_length            : 20
seq                           : 
refiner_mode                  : False
protein_seqs                  : AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKVQHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVNAPYAAALE
msa_mode                      : mmseqs
ligand_smiles                 : 
ligand_ccd                    : 
nucleic_type                  : dna
nucleic_seq                   : 
template_path                 : 
template_cif_chain_id         : 
diffuse_steps                 : 200
recycling_steps               : 3
boltz_model_version           : boltz2
boltz_model_path              : ~/.boltz/bo

/home/kyle/miniconda3/envs/proteinhunter/lib/python3.10/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.0.post0, which is newer than your current Lightning version: v2.5.0


✅ ProteinHunter_Boltz initialized.
Data dictionary:
 {'sequences': [{'protein': {'id': ['A'], 'sequence': 'X', 'msa': 'empty', 'cyclic': True}}, {'protein': {'id': ['B'], 'sequence': 'AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKVQHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVNAPYAAALE', 'msa': '/home/kyle/Protein-Hunter/outputs/notebook_PDL1_cyclic_peptide_no_af3_smoke/0_protein_hunter_design/B_env/msa.npz'}}]}

=== Starting Design Run 0/0 ===
Binder initial sequence length: 10
Empty MSA found; using single sequence mode.

--- Run 0, Cycle 1 ---
Empty MSA found; using single sequence mode.
ipTM: 0.90 pLDDT: 0.95 iPLDDT: 0.95 Alanine count: 0
✅ Saved run 0 cycle 1 YAML.
✅ Copied PDB to /home/kyle/Protein-Hunter/outputs/notebook_PDL1_cyclic_peptide_no_af3_smoke/high_iptm_pdb/notebook_PDL1_cyclic_peptide_no_af3_smoke_run_0_cycle_1_structure.pdb
✅ Appended high-ipTM metrics to /home/kyle/Protein-Hunter/outputs/notebook_PDL1_cyclic_peptide_no_af3_smoke/summary_high_iptm

## 4. Inspect outputs

Under `save_dir`: `summary_all_runs.csv`, and when thresholds are met, `high_iptm_yaml` / `high_iptm_pdb` (see README “Successful Designs”).

If `DOWNSTREAM_VALIDATION` was `"chai"`, also check `save_dir/1_af3_rosetta_validation/` for Chai holo/apo CIFs, converted PDBs, Rosetta `rosetta_energy.csv`, and filtered CSVs from the same Rosetta path as AF3 validation (folder name is historical).

In [4]:
import pandas as pd
from IPython.display import display

summary = OUT_DIR / "summary_all_runs.csv"
if summary.is_file():
    display(pd.read_csv(summary).head())
else:
    print("No summary yet:", summary)

for sub in ["high_iptm_yaml", "high_iptm_pdb", "high_iptm_cif", "1_af3_rosetta_validation"]:
    p = OUT_DIR / sub
    if p.is_dir():
        files = list(p.iterdir())
        print(f"{sub}: {len(files)} files")
    else:
        print(f"{sub}: (missing)")

,run_id,cycle_0_iptm,cycle_0_plddt,cycle_0_iplddt,cycle_0_alanine,cycle_0_seq,cycle_1_iptm,cycle_1_plddt,cycle_1_iplddt,cycle_1_alanine,...,cycle_2_seq,cycle_3_iptm,cycle_3_plddt,cycle_3_iplddt,cycle_3_alanine,cycle_3_seq,best_iptm,best_cycle,best_plddt,best_seq
0,0,0.168368,0.893903,0.824428,0,NaN,0.897397,0.948923,0.947889,0,...,RLVGGRLVDG,0.847805,0.947573,0.949519,0,RIVDGKLVDG,0.897397,1,0.948923,KLVGDELIDG


high_iptm_yaml: 3 files
high_iptm_pdb: 3 files
high_iptm_cif: (missing)
1_af3_rosetta_validation: 6 files
